In [0]:
%sql
SELECT COUNT(DISTINCT fk_commercial_id)
FROM (
    SELECT fk_commercial_id, COUNT(DISTINCT fk_client_id) AS dist_client_id
    FROM prod.detection.commercial_id_external_firehose
    GROUP BY 1
)
WHERE dist_client_id > 1

In [0]:
%sql
WITH kintiq_ads AS (
    SELECT fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose
    WHERE fk_client_id = 753
    GROUP BY 1
)
, non_kinetiq_ads AS (
    SELECT a.fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose AS a
    JOIN kintiq_ads AS b
       ON a.fk_commercial_id = b.fk_commercial_id
    WHERE a.fk_client_id != 753
)
SELECT COUNT(DISTINCT k.fk_commercial_id) AS kinetiq_ads
, COUNT(DISTINCT nk.fk_commercial_id) AS non_kinetiq_ads
FROM kintiq_ads k
LEFT JOIN non_kinetiq_ads nk
  ON nk.fk_commercial_id = k.fk_commercial_id

In [0]:
%sql
WITH springserve_ads AS (
    SELECT fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose
    WHERE fk_client_id = 1215
    GROUP BY 1
)
, non_springserve_ads AS (
    SELECT a.fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose AS a
    JOIN springserve_ads AS b
       ON a.fk_commercial_id = b.fk_commercial_id
    WHERE a.fk_client_id != 1215
)
SELECT COUNT(DISTINCT k.fk_commercial_id) AS kinetiq_ads
, COUNT(DISTINCT nk.fk_commercial_id) AS non_kinetiq_ads
FROM springserve_ads k
LEFT JOIN non_springserve_ads nk
  ON nk.fk_commercial_id = k.fk_commercial_id

In [0]:
%sql
WITH kintiq_ads AS (
    SELECT fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose
    WHERE fk_client_id = 753
    GROUP BY 1
)
, non_kinetiq_ads AS (
    SELECT a.fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose AS a
    JOIN kintiq_ads AS b
       ON a.fk_commercial_id = b.fk_commercial_id
    WHERE a.fk_client_id != 753
)
SELECT cief.*
FROM prod.detection.commercial_id_external_firehose cief
JOIN non_kinetiq_ads nk
  ON nk.fk_commercial_id = cief.fk_commercial_id
GROUP BY ALL
ORDER BY 1, 2

In [0]:
%sql
WITH kintiq_ads AS (
    SELECT fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose cief
    JOIN prod.detection.commercial_ids_firehose c
      ON c.commercial_id = cief.fk_commercial_id
    JOIN prod.public.fileingest_cidmap cid
      ON cid.reportingid = cief.external_id
    WHERE cief.fk_client_id = 753
    GROUP BY 1
)
, non_kinetiq_ads AS (
    SELECT cief.fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose AS cief
    JOIN prod.detection.commercial_ids_firehose c
      ON c.commercial_id = cief.fk_commercial_id
    JOIN kintiq_ads AS b
       ON cief.fk_commercial_id = b.fk_commercial_id
    JOIN prod.public.fileingest_cidmap cid
      ON cid.reportingid = cief.external_id
    WHERE cief.fk_client_id != 753
    GROUP BY ALL
)
-- SELECT COUNT(DISTINCT cief.fk_commercial_id)
-- SELECT cl.client_name, cief.*
FROM prod.detection.commercial_id_external_firehose cief
JOIN non_kinetiq_ads nk
  ON nk.fk_commercial_id = cief.fk_commercial_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
-- GROUP BY ALL
-- ORDER BY 2, 1

In [0]:
%sql
WITH kinads AS (
    WITH kintiq_ads AS (
        SELECT fk_commercial_id
        FROM prod.detection.commercial_id_external_firehose cief
        JOIN prod.detection.commercial_ids_firehose c
        ON c.commercial_id = cief.fk_commercial_id
        JOIN prod.public.fileingest_cidmap cid
        ON cid.reportingid = cief.external_id
        WHERE cief.fk_client_id = 753
        GROUP BY 1
    )
    , non_kinetiq_ads AS (
        SELECT cief.fk_commercial_id
        FROM prod.detection.commercial_id_external_firehose AS cief
        JOIN prod.detection.commercial_ids_firehose c
            ON c.commercial_id = cief.fk_commercial_id
        JOIN kintiq_ads AS b
            ON cief.fk_commercial_id = b.fk_commercial_id
        JOIN prod.public.fileingest_cidmap cid
            ON cid.reportingid = cief.external_id
        WHERE cief.fk_client_id != 753
        GROUP BY ALL
    )
    SELECT cief.fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose cief
    JOIN non_kinetiq_ads nk
    ON nk.fk_commercial_id = cief.fk_commercial_id
    GROUP BY ALL
)
, spingads AS (
    WITH springserve_ads AS (
        SELECT fk_commercial_id
        FROM prod.detection.commercial_id_external_firehose cief
        JOIN prod.detection.commercial_ids_firehose c
            ON c.commercial_id = cief.fk_commercial_id
        JOIN prod.public.fileingest_cidmap cid
            ON cid.reportingid = cief.external_id
        WHERE cief.fk_client_id = 1215
        GROUP BY 1
    )
    , non_springserve_ads AS (
        SELECT cief.fk_commercial_id
        FROM prod.detection.commercial_id_external_firehose AS cief
        JOIN prod.detection.commercial_ids_firehose c
            ON c.commercial_id = cief.fk_commercial_id
        JOIN springserve_ads AS b
            ON cief.fk_commercial_id = b.fk_commercial_id
        JOIN prod.public.fileingest_cidmap cid
            ON cid.reportingid = cief.external_id
        WHERE cief.fk_client_id != 1215
        GROUP BY 1
    )
    SELECT cief.fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose cief
    JOIN non_springserve_ads nk
        ON nk.fk_commercial_id = cief.fk_commercial_id
    GROUP BY ALL
)
SELECT 'Kinetiq' AS client, fk_commercial_id 
FROM kinads
UNION ALL
SELECT 'Springserve' AS client, fk_commercial_id 
FROM spingads

In [0]:
%sql
CREATE TABLE dev.mohit_gangwani.bad_kinetiq_ads AS
WITH kintiq_ads AS (
    SELECT fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose cief
    JOIN prod.detection.commercial_ids_firehose c
    ON c.commercial_id = cief.fk_commercial_id
    JOIN prod.public.fileingest_cidmap cid
    ON cid.reportingid = cief.external_id
    WHERE cief.fk_client_id = 753
    GROUP BY 1
)
, non_kinetiq_ads AS (
    SELECT cief.fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose AS cief
    JOIN prod.detection.commercial_ids_firehose c
        ON c.commercial_id = cief.fk_commercial_id
    JOIN kintiq_ads AS b
        ON cief.fk_commercial_id = b.fk_commercial_id
    JOIN prod.public.fileingest_cidmap cid
        ON cid.reportingid = cief.external_id
    WHERE cief.fk_client_id != 753
    GROUP BY ALL
)
SELECT cief.fk_commercial_id
FROM prod.detection.commercial_id_external_firehose cief
JOIN non_kinetiq_ads nk
ON nk.fk_commercial_id = cief.fk_commercial_id
GROUP BY ALL

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.tracking_bad_kinetiq_ids;
CREATE TABLE dev.mohit_gangwani.tracking_bad_kinetiq_ids AS
SELECT DATE_TRUNC('DAY', vc.partition_key) AS day
, SPLIT_PART(tvog_num, '_', 2) AS client_name
, COUNT(*) AS session_count
, COUNT(DISTINCT bad.fk_commercial_id) AS dist_comm_id_count
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN dev.mohit_gangwani.bad_kinetiq_ads bad
  ON bad.fk_commercial_id = vc.fk_commercial_id
WHERE vc.partition_key = CURRENT_DATE - 730
  AND vc.fk_commercial_source_id = 1
  AND vc.external_id IS NULL
GROUP BY 1, 2

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.tracking_bad_kinetiq_ids

In [0]:
for i in range(729, -1, -1):
    spark.sql(f"""
        INSERT INTO dev.mohit_gangwani.tracking_bad_kinetiq_ids
        SELECT DATE_TRUNC('DAY', vc.partition_key) AS day
        , SPLIT_PART(tvog_num, '_', 2) AS client_name
        , COUNT(*) AS session_count
        , COUNT(DISTINCT bad.fk_commercial_id) AS dist_comm_id_count
        FROM prod.detection.viewing_commercials_firehose_dedup vc
        JOIN dev.mohit_gangwani.bad_kinetiq_ads bad
            ON bad.fk_commercial_id = vc.fk_commercial_id
        WHERE vc.partition_key = CURRENT_DATE - {i}
            AND vc.fk_commercial_source_id = 1
            AND vc.external_id IS NULL
        GROUP BY 1, 2;
    """)
    print(f"done with {i}")

In [0]:
%sql
SELECT client_name, dist_comm_id_count, SUM(session_count) FROM dev.mohit_gangwani.tracking_bad_kinetiq_ids
GROUP BY 1, 2

In [0]:
%sql
SELECT * FROM prod.detection.commercial_ids_firehose
WHERE commercial_id IN (100940936,100833172)

In [0]:
%sql
SELECT * FROM prod.detection.commercial_ids_firehose
WHERE commercial_cid LIKE 'tatari-1761339772-%'

In [0]:
    %sql
SELECT SUBSTRING(cid, 20) FROM prod.public.fileingest_cidmap
WHERE reportingid IN ('GGHACDV115H', 'AE16546-2025-36-14708')

In [0]:
%sql
SELECT fk_commercial_id
FROM prod.detection.commercial_id_external_firehose
WHERE fk_client_id = 753
GROUP BY 1
ORDER BY 1 DESC
LIMIT 100

In [0]:
%sql
WITH springserve_ads AS (
    SELECT fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose
    WHERE fk_client_id = 1215
    GROUP BY 1
)
, non_springserve_ads AS (
    SELECT a.fk_commercial_id
    FROM prod.detection.commercial_id_external_firehose AS a
    JOIN springserve_ads AS b
       ON a.fk_commercial_id = b.fk_commercial_id
    WHERE a.fk_client_id != 1215
)
SELECT cief.*
FROM prod.detection.commercial_id_external_firehose cief
JOIN non_springserve_ads nk
  ON nk.fk_commercial_id = cief.fk_commercial_id
GROUP BY ALL
ORDER BY 1, 2

In [0]:
%sql
SELECT * FROM prod.detection.clients
WHERE client_name ilike 'spring%'

In [0]:
%sql
SELECT COUNT(DISTINCT fk_commercial_id)
FROM prod.detection.commercial_id_external_firehose

In [0]:
%sql
SELECT COUNT(commercial_id)
FROM (
    SELECT commercial_id, COUNT(DISTINCT commercial_cid) AS dist_cid
    FROM prod.detection.commercial_ids_firehose
    GROUP BY 1
)
WHERE dist_cid > 1
-- WHERE fk_external_id IS NOT NULL
LIMIT 100

In [0]:
%sql
SELECT fk_commercial_id AS current_bad_comm_id
, commercial_id AS new_correct_id
, external_id
, client_name AS client_name_in_external_table
, customer AS client_name_in_fileingest
, cid AS fileingest_cid
, commercial_cid
, fk_client_id
FROM (
    SELECT *
    , ROW_NUMBER() OVER (PARTITION BY fk_commercial_id, client_name ORDER BY NVL(commercial_id, 1) DESC) AS rn
    FROM(
        WITH kin_id AS (
            WITH kintiq_ads AS (
                SELECT fk_commercial_id
                FROM prod.detection.commercial_id_external_firehose cief
                WHERE cief.fk_client_id = 753
                GROUP BY 1
            )
            , non_kinetiq_ads AS (
                SELECT cief.fk_commercial_id
                FROM prod.detection.commercial_id_external_firehose AS cief
                JOIN kintiq_ads AS b
                    ON cief.fk_commercial_id = b.fk_commercial_id
                WHERE cief.fk_client_id != 753
                GROUP BY ALL
            )
            SELECT cief.fk_commercial_id, cief.external_id, cief.fk_client_id
            FROM prod.detection.commercial_id_external_firehose cief
            JOIN non_kinetiq_ads nk
                ON nk.fk_commercial_id = cief.fk_commercial_id
            GROUP BY ALL
        )
        SELECT kin_id.fk_commercial_id
        , cid.commercial_id
        , cid.commercial_cid
        , kin_id.external_id
        , cl.client_name
        , fim.customer
        , fim.cid
        , kin_id.fk_client_id
        FROM kin_id
        JOIN prod.detection.clients cl
            ON cl.client_id = kin_id.fk_client_id
        LEFT JOIN prod.public.cid_map fim
            ON fim.value = kin_id.external_id
        LEFT JOIN prod.detection.commercial_ids_firehose cid
            ON cid.commercial_cid = SUBSTRING(fim.cid, 20)
        GROUP BY ALL)
        -- WHERE client_id NOT IN ('kinetiq', 'kintest')
        -- AND client_name = client_id
)
WHERE rn = 1
ORDER BY 1, 2

In [0]:
%sql
SELECT fk_commercial_id AS current_bad_comm_id
, commercial_id AS new_correct_id
, external_id
, client_name AS client_name_in_external_table
, client_id AS client_name_in_fileingest
, cid AS fileingest_cid
, commercial_cid
, fk_client_id
FROM (
    SELECT *
    , ROW_NUMBER() OVER (PARTITION BY fk_commercial_id, client_name ORDER BY NVL(commercial_id, 1) DESC) AS rn
    FROM(
        WITH kin_id AS (
            WITH kintiq_ads AS (
                SELECT fk_commercial_id
                FROM prod.detection.commercial_id_external_firehose cief
                WHERE cief.fk_client_id = 1215
                GROUP BY 1
            )
            , non_kinetiq_ads AS (
                SELECT cief.fk_commercial_id
                FROM prod.detection.commercial_id_external_firehose AS cief
                JOIN kintiq_ads AS b
                    ON cief.fk_commercial_id = b.fk_commercial_id
                WHERE cief.fk_client_id != 1215
                GROUP BY ALL
            )
            SELECT cief.fk_commercial_id, cief.external_id, cief.fk_client_id
            FROM prod.detection.commercial_id_external_firehose cief
            JOIN non_kinetiq_ads nk
                ON nk.fk_commercial_id = cief.fk_commercial_id
            GROUP BY ALL
        )
        SELECT kin_id.fk_commercial_id
        , cid.commercial_id
        , cid.commercial_cid
        , kin_id.external_id
        , cl.client_name
        , COALESCE(fim.client_id, cid_map.customer) AS client_id
        , COALESCE(fim.cid, cid_map.cid) AS cid
        , kin_id.fk_client_id
        FROM kin_id
        JOIN prod.detection.clients cl
            ON cl.client_id = kin_id.fk_client_id
        LEFT JOIN prod.public.fileingest_cidmap fim
            ON fim.reportingid = kin_id.external_id
        LEFT JOIN prod.public.cid_map
          ON cid_map.value = kin_id.external_id
        LEFT JOIN prod.detection.commercial_ids_firehose cid
            ON cid.commercial_cid = COALESCE(SUBSTRING(fim.cid, 20), SUBSTRING(cid_map.cid, 20))
        GROUP BY ALL)
        WHERE TRUE
        -- AND (client_id NOT IN ('SpringServe-Prod') OR client_id IS NULL)
        -- AND client_name = client_id
)
-- WHERE rn = 1
ORDER BY 1, 2

In [0]:
%sql
SELECT * FROM prod.public.fileingest_cidmap
WHERE contenttitle ILIKE 'Pudding Plate%'
-- WHERE reportingid ILIKE 'TYTQ%'
-- WHERE client_id = 'kantar'

In [0]:
%sql
SELECT * FROM prod.detection.commercial_id_external_firehose
WHERE external_id ILIKE 'CIPT22%'

In [0]:
%sql
-- DROP TABLE IF EXISTS dev.mohit_gangwani.duplicate_ads_mapping_table;
-- CREATE TABLE dev.mohit_gangwani.duplicate_ads_mapping_table AS
WITH springserve_ads AS (
    SELECT fk_commercial_id AS current_bad_comm_id
    , commercial_id AS new_correct_id
    , external_id
    , fk_client_id
    -- SELECT *
    FROM (
        SELECT *
        , ROW_NUMBER() OVER (PARTITION BY fk_commercial_id, client_name ORDER BY NVL(commercial_id, 1) DESC) AS rn
        FROM(
            WITH sprs_id AS (
                WITH springserve_ads AS (
                    SELECT fk_commercial_id
                    FROM prod.detection.commercial_id_external_firehose cief
                    WHERE cief.fk_client_id = 1215
                    GROUP BY 1
                )
                , non_springserve_ads AS (
                    SELECT cief.fk_commercial_id
                    FROM prod.detection.commercial_id_external_firehose AS cief
                    JOIN springserve_ads AS b
                        ON cief.fk_commercial_id = b.fk_commercial_id
                    WHERE cief.fk_client_id != 1215
                    GROUP BY ALL
                )
                SELECT cief.fk_commercial_id, cief.external_id, cief.fk_client_id
                FROM prod.detection.commercial_id_external_firehose cief
                JOIN non_springserve_ads nk
                    ON nk.fk_commercial_id = cief.fk_commercial_id
                GROUP BY ALL
            )
            SELECT sprs_id.fk_commercial_id
            , cid.commercial_id
            , cid.commercial_cid
            , sprs_id.external_id
            , cl.client_name
            , COALESCE(fim.client_id, cid_map.customer) AS client_id
            , COALESCE(fim.cid, cid_map.cid) AS cid
            , sprs_id.fk_client_id
            FROM sprs_id
            JOIN prod.detection.clients cl
                ON cl.client_id = sprs_id.fk_client_id
            LEFT JOIN prod.public.fileingest_cidmap fim
                ON fim.reportingid = sprs_id.external_id
            LEFT JOIN prod.public.cid_map
                ON cid_map.value = sprs_id.external_id
            LEFT JOIN prod.detection.commercial_ids_firehose cid
                ON cid.commercial_cid = COALESCE(SUBSTRING(fim.cid, 20), SUBSTRING(cid_map.cid, 20))
            GROUP BY ALL)
            WHERE TRUE
              AND client_id NOT IN ('SpringServe-Prod')
              AND client_name = client_id
              AND fk_commercial_id != commercial_id
    )
    WHERE rn = 1
    ORDER BY 1, 2
)
, kinetiq_ads AS (
    SELECT fk_commercial_id AS current_bad_comm_id
    , commercial_id AS new_correct_id
    , external_id
    , fk_client_id
    -- SELECT *
    FROM (
        SELECT *
        , ROW_NUMBER() OVER (PARTITION BY fk_commercial_id, client_name ORDER BY NVL(commercial_id, 1) DESC) AS rn
        FROM(
            WITH kin_id AS (
                WITH kintiq_ads AS (
                    SELECT fk_commercial_id
                    FROM prod.detection.commercial_id_external_firehose cief
                    WHERE cief.fk_client_id = 753
                    GROUP BY 1
                )
                , non_kinetiq_ads AS (
                    SELECT cief.fk_commercial_id
                    FROM prod.detection.commercial_id_external_firehose AS cief
                    JOIN kintiq_ads AS b
                        ON cief.fk_commercial_id = b.fk_commercial_id
                    WHERE cief.fk_client_id != 753
                    GROUP BY ALL
                )
                SELECT cief.fk_commercial_id, cief.external_id, cief.fk_client_id
                FROM prod.detection.commercial_id_external_firehose cief
                JOIN non_kinetiq_ads nk
                    ON nk.fk_commercial_id = cief.fk_commercial_id
                GROUP BY ALL
            )
            SELECT kin_id.fk_commercial_id
            , cid.commercial_id
            , cid.commercial_cid
            , kin_id.external_id
            , cl.client_name
            , COALESCE(fim.client_id, cid_map.customer) AS client_id
            , COALESCE(fim.cid, cid_map.cid) AS cid
            , kin_id.fk_client_id
            FROM kin_id
            JOIN prod.detection.clients cl
                ON cl.client_id = kin_id.fk_client_id
            LEFT JOIN prod.public.fileingest_cidmap fim
                ON fim.reportingid = kin_id.external_id
            LEFT JOIN prod.public.cid_map
                ON cid_map.value = kin_id.external_id
            LEFT JOIN prod.detection.commercial_ids_firehose cid
                ON cid.commercial_cid = COALESCE(SUBSTRING(fim.cid, 20), SUBSTRING(cid_map.cid, 20))
            GROUP BY ALL)
            WHERE TRUE
              AND client_id NOT IN ('kinetiq')
              AND client_name = client_id
              AND fk_commercial_id != commercial_id
    )
    WHERE rn = 1
    ORDER BY 1, 2
)
SELECT * FROM springserve_ads
UNION
SELECT * FROM kinetiq_ads
ORDER BY 1, 2, 3, 4

In [0]:
%sql
SELECT MAX(NULLIF(LEAST(NULL, NULL, 'XYZ'), 'XYZ'))

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.duplicate_ads_deletion_table;
CREATE TABLE dev.mohit_gangwani.duplicate_ads_deletion_table AS
WITH springserve_ads AS (
    SELECT *
    FROM (
        SELECT *
        , ROW_NUMBER() OVER (PARTITION BY fk_commercial_id, client_name ORDER BY NVL(commercial_id, 1) DESC) AS rn
        FROM(
            WITH sprs_id AS (
                WITH springserve_ads AS (
                    SELECT fk_commercial_id
                    FROM prod.detection.commercial_id_external_firehose cief
                    WHERE cief.fk_client_id = 1215
                    GROUP BY 1
                )
                , non_springserve_ads AS (
                    SELECT cief.fk_commercial_id
                    FROM prod.detection.commercial_id_external_firehose AS cief
                    JOIN springserve_ads AS b
                        ON cief.fk_commercial_id = b.fk_commercial_id
                    WHERE cief.fk_client_id != 1215
                    GROUP BY ALL
                )
                SELECT cief.fk_commercial_id, cief.external_id, cief.fk_client_id
                FROM prod.detection.commercial_id_external_firehose cief
                JOIN non_springserve_ads nk
                    ON nk.fk_commercial_id = cief.fk_commercial_id
                GROUP BY ALL
            )
            SELECT sprs_id.fk_commercial_id
            , CASE WHEN cid.commercial_id = sprs_id.fk_commercial_id THEN NULL ELSE cid.commercial_id END AS commercial_id
            , cid.commercial_cid
            , sprs_id.external_id
            , cl.client_name
            , fim.client_id
            , fim.cid
            , sprs_id.fk_client_id
            FROM sprs_id
            JOIN prod.detection.clients cl
                ON cl.client_id = sprs_id.fk_client_id
            LEFT JOIN prod.public.fileingest_cidmap fim
                ON fim.reportingid = sprs_id.external_id
            LEFT JOIN prod.detection.commercial_ids_firehose cid
                ON cid.commercial_cid = SUBSTRING(fim.cid, 20)
            GROUP BY ALL)
    )
    WHERE rn = 1
    ORDER BY 1, 2
)
, kinetiq_ads AS (
    SELECT *
    FROM (
        SELECT *
        , ROW_NUMBER() OVER (PARTITION BY fk_commercial_id, client_name ORDER BY NVL(commercial_id, 1) DESC) AS rn
        FROM(
            WITH kin_id AS (
                WITH kintiq_ads AS (
                    SELECT fk_commercial_id
                    FROM prod.detection.commercial_id_external_firehose cief
                    WHERE cief.fk_client_id = 753
                    GROUP BY 1
                )
                , non_kinetiq_ads AS (
                    SELECT cief.fk_commercial_id
                    FROM prod.detection.commercial_id_external_firehose AS cief
                    JOIN kintiq_ads AS b
                        ON cief.fk_commercial_id = b.fk_commercial_id
                    WHERE cief.fk_client_id != 753
                    GROUP BY ALL
                )
                SELECT cief.fk_commercial_id, cief.external_id, cief.fk_client_id
                FROM prod.detection.commercial_id_external_firehose cief
                JOIN non_kinetiq_ads nk
                    ON nk.fk_commercial_id = cief.fk_commercial_id
                GROUP BY ALL
            )
            SELECT kin_id.fk_commercial_id
            , CASE WHEN cid.commercial_id = kin_id.fk_commercial_id THEN NULL ELSE cid.commercial_id END AS commercial_id
            , cid.commercial_cid
            , kin_id.external_id
            , cl.client_name
            , fim.client_id
            , fim.cid
            , kin_id.fk_client_id
            FROM kin_id
            JOIN prod.detection.clients cl
                ON cl.client_id = kin_id.fk_client_id
            LEFT JOIN prod.public.fileingest_cidmap fim
                ON fim.reportingid = kin_id.external_id
            LEFT JOIN prod.detection.commercial_ids_firehose cid
                ON cid.commercial_cid = SUBSTRING(fim.cid, 20)
            GROUP BY ALL)
    )
    WHERE rn = 1
    ORDER BY 1, 2
)
, all_ads AS (
SELECT fk_commercial_id AS current_bad_comm_id
    , external_id
    , fk_client_id
    FROM springserve_ads
WHERE client_id IS NULL OR client_id NOT IN ('SpringServe-Prod')
UNION
SELECT fk_commercial_id AS current_bad_comm_id
    , external_id
    , fk_client_id
    FROM kinetiq_ads
WHERE client_id IS NULL OR client_id NOT IN ('kinetiq', 'kinetiqvideo')
)
SELECT a.*
FROM all_ads a
LEFT JOIN dev.mohit_gangwani.duplicate_ads_mapping_table x
  ON a.current_bad_comm_id = x.current_bad_comm_id
WHERE x.current_bad_comm_id IS NULL
GROUP BY ALL
ORDER BY 1, 2, 3


In [0]:
%sql
SELECT COUNT(*) FROM dev.mohit_gangwani.duplicate_ads_mapping_table

In [0]:
%sql
SELECT COUNT(*) FROM dev.mohit_gangwani.duplicate_ads_deletion_table

In [0]:
%sql
SELECT * FROM prod.public.cid_map
JOIN dev.mohit_gangwani.duplicate_ads_mapping_table dev
  ON cid_map.cid
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.temp_commercial_id_external_firehose;
CREATE TABLE dev.mohit_gangwani.temp_commercial_id_external_firehose AS
SELECT * FROM prod.detection.commercial_id_external_firehose;

In [0]:
%sql
-- Step 1: Update fk_commercial_id to new_correct_id in commercial_id_external_firehose
MERGE INTO dev.mohit_gangwani.temp_commercial_id_external_firehose AS target
USING dev.mohit_gangwani.duplicate_ads_mapping_table AS source
  ON target.fk_commercial_id = source.current_bad_comm_id
  AND target.fk_client_id = source.fk_client_id
WHEN MATCHED THEN UPDATE SET fk_commercial_id = source.new_correct_id;

In [0]:
%sql
-- Step 2: Delete rows from commercial_id_external_firehose using deletion table
MERGE INTO dev.mohit_gangwani.temp_commercial_id_external_firehose AS target
USING dev.mohit_gangwani.duplicate_ads_deletion_table AS source
  ON target.fk_commercial_id = source.current_bad_comm_id
  AND target.fk_client_id = source.fk_client_id
WHEN MATCHED THEN DELETE;

In [0]:
%sql
WITH springserve_ads AS (
    SELECT *
    FROM (
        SELECT *
        FROM(
            WITH sprs_id AS (
                WITH springserve_ads AS (
                    SELECT fk_commercial_id
                    FROM dev.mohit_gangwani.temp_commercial_id_external_firehose cief
                    WHERE cief.fk_client_id = 1215
                    GROUP BY 1
                )
                , non_springserve_ads AS (
                    SELECT cief.fk_commercial_id
                    FROM dev.mohit_gangwani.temp_commercial_id_external_firehose AS cief
                    JOIN springserve_ads AS b
                        ON cief.fk_commercial_id = b.fk_commercial_id
                    WHERE cief.fk_client_id != 1215
                    GROUP BY ALL
                )
                SELECT cief.fk_commercial_id, cief.external_id, cief.fk_client_id
                FROM dev.mohit_gangwani.temp_commercial_id_external_firehose cief
                JOIN non_springserve_ads nk
                    ON nk.fk_commercial_id = cief.fk_commercial_id
                GROUP BY ALL
            )
            SELECT sprs_id.fk_commercial_id
            , CASE WHEN cid.commercial_id = sprs_id.fk_commercial_id THEN NULL ELSE cid.commercial_id END AS commercial_id
            , cid.commercial_cid
            , sprs_id.external_id
            , cl.client_name
            , fim.client_id
            , fim.cid
            , sprs_id.fk_client_id
            FROM sprs_id
            JOIN prod.detection.clients cl
                ON cl.client_id = sprs_id.fk_client_id
            LEFT JOIN prod.public.fileingest_cidmap fim
                ON fim.reportingid = sprs_id.external_id
            LEFT JOIN prod.detection.commercial_ids_firehose cid
                ON cid.commercial_cid = SUBSTRING(fim.cid, 20)
            GROUP BY ALL)
    )
    ORDER BY 1, 2
)
, kinetiq_ads AS (
    SELECT *
    FROM (
        SELECT *
        FROM(
            WITH kin_id AS (
                WITH kintiq_ads AS (
                    SELECT fk_commercial_id
                    FROM dev.mohit_gangwani.temp_commercial_id_external_firehose cief
                    WHERE cief.fk_client_id = 753
                    GROUP BY 1
                )
                , non_kinetiq_ads AS (
                    SELECT cief.fk_commercial_id
                    FROM dev.mohit_gangwani.temp_commercial_id_external_firehose AS cief
                    JOIN kintiq_ads AS b
                        ON cief.fk_commercial_id = b.fk_commercial_id
                    WHERE cief.fk_client_id != 753
                    GROUP BY ALL
                )
                SELECT cief.fk_commercial_id, cief.external_id, cief.fk_client_id
                FROM dev.mohit_gangwani.temp_commercial_id_external_firehose cief
                JOIN non_kinetiq_ads nk
                    ON nk.fk_commercial_id = cief.fk_commercial_id
                GROUP BY ALL
            )
            SELECT kin_id.fk_commercial_id
            , CASE WHEN cid.commercial_id = kin_id.fk_commercial_id THEN NULL ELSE cid.commercial_id END AS commercial_id
            , cid.commercial_cid
            , kin_id.external_id
            , cl.client_name
            , fim.client_id
            , fim.cid
            , kin_id.fk_client_id
            FROM kin_id
            JOIN prod.detection.clients cl
                ON cl.client_id = kin_id.fk_client_id
            LEFT JOIN prod.public.fileingest_cidmap fim
                ON fim.reportingid = kin_id.external_id
            LEFT JOIN prod.detection.commercial_ids_firehose cid
                ON cid.commercial_cid = SUBSTRING(fim.cid, 20)
            GROUP BY ALL)
    )
    ORDER BY 1, 2
)
SELECT fk_commercial_id AS current_bad_comm_id
    , external_id
    , fk_client_id
    FROM springserve_ads
UNION
SELECT fk_commercial_id AS current_bad_comm_id
    , external_id
    , fk_client_id
    FROM kinetiq_ads

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.temp_viewing_commercials_dedup_golden;
CREATE TABLE dev.mohit_gangwani.temp_viewing_commercials_dedup_golden AS
SELECT *
FROM prod.detection.viewing_commercials_dedup_golden
WHERE session_start >= '2026-05-01 00:00:00'
  AND session_start < '2026-05-02 00:00:00'
  AND session_start_hour >= '2026-05-01 00:00:00'
  AND session_start_hour < '2026-05-02 00:00:00'

In [0]:
%sql
-- Step 3: Delete rows where fk_commercial_source_id = 2 and client_name is not 'kinetiq'
DELETE FROM dev.mohit_gangwani.temp_viewing_commercials_dedup_golden
WHERE fk_commercial_source_id = 2
  AND commercial_client != 'kinetiq';

In [0]:
%sql
-- Step 4: Delete rows where fk_commercial_source_id = 3 and client_name is not 'springserve'
DELETE FROM dev.mohit_gangwani.temp_viewing_commercials_dedup_golden
WHERE fk_commercial_source_id = 3
  AND commercial_client != 'SpringServe-Prod';

In [0]:
%sql
DELETE FROM dev.mohit_gangwani.temp_viewing_commercials_dedup_golden
WHERE fk_commercial_id IN (
    SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_mapping_table
    UNION
    SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_deletion_table
)
AND commercial_client NOT IN ('kinetiq', 'SpringServe-Prod');

In [0]:
%sql
SELECT commercial_client, COUNT(*)
FROM prod.detection.viewing_commercials_golden vc
JOIN (
    select current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_mapping_table
    UNION
    SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_deletion_table
    ) c
  ON c.current_bad_comm_id = vc.fk_commercial_id
WHERE session_start >= '2026-06-01 00:00:00'
  AND session_start < '2026-06-02 00:00:00'
  AND session_start_hour >= '2026-06-01 00:00:00'
  AND session_start_hour < '2026-06-02 00:00:00'
  AND mod(vc.fk_tvid, 100) = 1
  -- AND fk_commercial_source_id = 1
GROUP BY 1

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.temp_viewing_commercials_golden;
CREATE TABLE dev.mohit_gangwani.temp_viewing_commercials_golden AS
SELECT *
FROM prod.detection.viewing_commercials_golden
WHERE session_start >= '2026-05-01 00:00:00'
  AND session_start < '2026-05-02 00:00:00'
  AND session_start_hour >= '2026-05-01 00:00:00'
  AND session_start_hour < '2026-05-02 00:00:00'
  AND MOD(fk_tvid, 100) = 1

In [0]:
%sql
DELETE FROM dev.mohit_gangwani.temp_viewing_commercials_golden
WHERE fk_commercial_id IN (
    SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_mapping_table
    UNION
    SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_deletion_table
)
AND commercial_client NOT IN ('kinetiq', 'SpringServe-Prod');

In [0]:
%sql
SELECT split_part(tvog_num, '_', 2)
, COUNT(*)
FROM stage.detection.viewing_commercials_firehose_dedup vc
JOIN (
    select current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_mapping_table
    UNION
    SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_deletion_table
    ) cief
  ON cief.current_bad_comm_id = vc.fk_commercial_id
GROUP BY 1

In [0]:
%sql
-- Step 7: Delete sessions from viewing_commercials_firehose_dedup where the bad commercial ID
-- is not assigned to kinetiq or springserve-prod
DELETE FROM prod.detection.viewing_commercials_firehose_dedup
WHERE fk_commercial_id IN (
    SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_mapping_table
    UNION
    SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_deletion_table
)
AND LOWER(SPLIT_PART(tvog_num, '_', 2)) NOT IN ('kinetiq', 'springserve-prod');

In [0]:
%sql
SELECT commercial_client, COUNT(*)
FROM prod.detection.viewing_commercials_golden vc
JOIN (
    select current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_mapping_table
    UNION
    SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_deletion_table
    ) cief
  ON cief.current_bad_comm_id = vc.fk_commercial_id
WHERE session_start_hour >= CURRENT_DATE
GROUP BY 1

In [0]:
%sql
SELECT commercial_client, COUNT(*)
FROM dev.mohit_gangwani.temp_viewing_commercials_dedup_golden vc
JOIN (
    select current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_mapping_table
    UNION
    SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_deletion_table
    ) cief
  ON cief.current_bad_comm_id = vc.fk_commercial_id
GROUP BY 1

In [0]:
%sql
SELECT cief.external_id, COUNT(*)
FROM prod.detection.viewing_commercials_cfe_golden vc
JOIN dev.mohit_gangwani.duplicate_ads_mapping_table cief
  ON cief.current_bad_comm_id = vc.fk_commercial_id
 AND cief.external_id = vc.external_id
WHERE session_start_hour >= CURRENT_DATE
GROUP BY 1

In [0]:
%sql
SELECT * FROM prod.detection.commercial_id_external_firehose
WHERE external_id = 'AE16546-2024-35-01130'

In [0]:
%sql
SELECT x.* FROM dev.mohit_gangwani.duplicate_ads_deletion_table x
  JOIN prod.detection.commercial_id_external_firehose cief
    ON cief.fk_commercial_id = x.current_bad_comm_id
    AND cief.external_id = x.external_id
    AND cief.fk_client_id IN (753, 1215)
    GROUP BY ALL

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.temp_comm_id_ext_sub_query_1;
CREATE TABLE dev.mohit_gangwani.temp_comm_id_ext_sub_query_1 AS
SELECT 
fc.cid,
fc.client_id AS customer,
fc.reportingid AS reportingid, 
fc.contenttitle AS title, 
fc.content_duration AS duration, 
fc.brand AS brand_name,
fc.vertical AS commercial_category 
FROM prod.public.fileingest_cidmap AS fc
WHERE fc.contenttype != 'content'

In [0]:
%sql
WITH dups AS (
    SELECT cid
    FROM (
        SELECT cid, COUNT(DISTINCT customer) AS cust_count
        FROM prod.public.cid_map
        GROUP BY 1
    ) c
    WHERE cust_count > 1
    GROUP BY 1
)
SELECT cid.*
FROM prod.public.cid_map cid
JOIN dups
  ON dups.cid = cid.cid
WHERE customer != 'ispot'
ORDER BY cid, created_at DESC
LIMIT 100


In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.temp_comm_id_ext_sub_query_2;
CREATE TABLE dev.mohit_gangwani.temp_comm_id_ext_sub_query_2 AS
SELECT 
cp.cid
, cp.customer AS customer
, cp.value AS reportingid
, im.title AS title
, TRY_CAST(im.duration AS BIGINT) AS duration
, im.brand_name AS brand_name
, CASE WHEN ii.parent_industry_name IS NOT NULL THEN (ii.parent_industry_name) || ' / ' || ii.industry_name ELSE NULL END  as commercial_category
FROM prod.public.cid_map AS cp
LEFT OUTER JOIN prod.public.ispot_ad_overview_metadata AS im ON TRY_CAST(cp.value AS STRING) = TRY_CAST(im.id AS STRING)
LEFT OUTER JOIN prod.public.ispot_industries AS ii ON TRY_CAST(cp.value AS STRING) = TRY_CAST(ii.id AS STRING)
-- UNION
-- SELECT *
-- FROM dev.mohit_gangwani.temp_comm_id_ext_sub_query_1

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.temp_comm_id_ext_thirdSubQuery;
CREATE TABLE dev.mohit_gangwani.temp_comm_id_ext_thirdSubQuery AS
SELECT DISTINCT
commercial_id,
client_id,
reportingid,
title,
duration,
brand_name,
commercial_category
FROM dev.mohit_gangwani.temp_comm_id_ext_sub_query_2 AS cids
INNER JOIN prod.detection.clients AS cl ON cids.customer = cl.client_name
INNER JOIN prod.detection.commercial_ids_firehose AS ci ON SUBSTRING(cids.cid, 20) = ci.commercial_cid
ORDER BY commercial_id, client_id, reportingid;

In [0]:
%sql
SELECT ci.* FROM dev.mohit_gangwani.temp_comm_id_ext_thirdSubQuery t
JOIN prod.detection.commercial_ids_firehose AS ci
  ON ci.commercial_id = t.commercial_id
ORDER BY ci.created_at DESC
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.temp_commercial_id_external_firehose;
CREATE TABLE dev.mohit_gangwani.temp_commercial_id_external_firehose AS
SELECT * FROM prod.detection.commercial_id_external_firehose;

In [0]:
%sql
    DELETE FROM dev.mohit_gangwani.temp_commercial_id_external_firehose AS commercial_id_external_firehose
    WHERE NOT EXISTS (SELECT 1 
                      FROM  dev.mohit_gangwani.temp_comm_id_ext_thirdSubQuery AS cid
                        WHERE commercial_id_external_firehose.fk_commercial_id = cid.commercial_id
                        AND commercial_id_external_firehose.fk_client_id = cid.client_id
                        AND commercial_id_external_firehose.external_id = cid.reportingid
                        AND IFNULL(commercial_id_external_firehose.brand_name,'xxxxx') = IFNULL(cid.brand_name,'xxxxx')
                        AND IFNULL(commercial_id_external_firehose.title,'xxxxx') = IFNULL(cid.title,'xxxxx')
                        AND IFNULL(commercial_id_external_firehose.commercial_category,'xxxxx') = IFNULL(cid.commercial_category,'xxxxx')
                        AND commercial_id_external_firehose.duration = cid.duration
                        AND commercial_id_external_firehose.source = 'Ingested')
    AND commercial_id_external_firehose.source =  'Ingested'

In [0]:
%sql
SELECT 1 
                      FROM prod.detection.commercial_id_external_firehose AS cie 
                      WHERE 
                        LIMIT 100

In [0]:
%sql
    INSERT INTO dev.mohit_gangwani.temp_commercial_id_external_firehose
      SELECT 
      commercial_id,
      client_id,
      reportingid,
      title,
      duration,
      brand_name,
      commercial_category,
      'Ingested'
    FROM dev.mohit_gangwani.temp_comm_id_ext_thirdSubQuery AS cid
    WHERE NOT EXISTS (SELECT 1 
                      FROM dev.mohit_gangwani.temp_commercial_id_external_firehose AS cie 
                      WHERE cie.fk_commercial_id=cid.commercial_id
                        AND cie.fk_client_id=cid.client_id
                        AND cie.external_id=cid.reportingid
                        AND IFNULL(cie.brand_name,'xxxxx')=IFNULL(cid.brand_name,'xxxxx')
                        AND IFNULL(cie.title,'xxxxx')=IFNULL(cid.title,'xxxxx')
                        AND IFNULL(cie.commercial_category,'xxxxx')=IFNULL(cid.commercial_category,'xxxxx')
                        AND cie.duration = cid.duration
                        )

In [0]:
%sql
-- Step 1: Update fk_commercial_id to new_correct_id in commercial_id_external_firehose
MERGE INTO dev.mohit_gangwani.temp_commercial_id_external_firehose AS target
USING dev.mohit_gangwani.duplicate_ads_mapping_table AS source
  ON target.fk_commercial_id = source.current_bad_comm_id
  AND target.fk_client_id = source.fk_client_id
WHEN MATCHED THEN UPDATE SET fk_commercial_id = source.new_correct_id;

In [0]:
%sql
-- Step 2: Delete rows from commercial_id_external_firehose using deletion table
MERGE INTO dev.mohit_gangwani.temp_commercial_id_external_firehose AS target
USING dev.mohit_gangwani.duplicate_ads_deletion_table AS source
  ON target.fk_commercial_id = source.current_bad_comm_id
  AND target.fk_client_id = source.fk_client_id
WHEN MATCHED THEN DELETE;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.temp_comm_id_ext_thirdSubQuery
WHERE commercial_id IN (
    SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_mapping_table
    UNION
    SELECT current_bad_comm_id FROM dev.mohit_gangwani.duplicate_ads_deletion_table
)
ORDER BY 1, 2, 3

In [0]:
%sql
WITH springserve_ads AS (
    -- SELECT fk_commercial_id AS current_bad_comm_id
    -- , commercial_id AS new_correct_id
    -- , external_id
    -- , fk_client_id
    SELECT *
    FROM (
    --     SELECT *
    --     , ROW_NUMBER() OVER (PARTITION BY fk_commercial_id, client_name ORDER BY NVL(commercial_id, 1) DESC) AS rn
    --     FROM(
            WITH sprs_id AS (
                WITH springserve_ads AS (
                    SELECT fk_commercial_id
                    FROM prod.detection.commercial_id_external_firehose cief
                    WHERE cief.fk_client_id = 1215
                    GROUP BY 1
                )
                , non_springserve_ads AS (
                    SELECT cief.fk_commercial_id
                    FROM prod.detection.commercial_id_external_firehose AS cief
                    JOIN springserve_ads AS b
                        ON cief.fk_commercial_id = b.fk_commercial_id
                    WHERE cief.fk_client_id != 1215
                    GROUP BY ALL
                )
                SELECT cief.fk_commercial_id, cief.external_id, cief.fk_client_id
                FROM prod.detection.commercial_id_external_firehose cief
                JOIN non_springserve_ads nk
                    ON nk.fk_commercial_id = cief.fk_commercial_id
                GROUP BY ALL
            )
            SELECT sprs_id.fk_commercial_id
            , cid.commercial_id
            , cid.commercial_cid
            , sprs_id.external_id
            , cl.client_name
            , fim.client_id
            , fim.cid
            , sprs_id.fk_client_id
            FROM sprs_id
            JOIN prod.detection.clients cl
                ON cl.client_id = sprs_id.fk_client_id
            LEFT JOIN prod.public.fileingest_cidmap fim
                ON fim.reportingid = sprs_id.external_id
            LEFT JOIN prod.detection.commercial_ids_firehose cid
                ON cid.commercial_cid = SUBSTRING(fim.cid, 20)
            GROUP BY ALL)
            WHERE TRUE
            --   AND client_id NOT IN ('SpringServe-Prod')
            --   AND client_name = client_id
            --   AND fk_commercial_id != commercial_id
    -- )
    -- WHERE rn = 1
    -- ORDER BY 1, 2
)
, kinetiq_ads AS (
    -- SELECT fk_commercial_id AS current_bad_comm_id
    -- , commercial_id AS new_correct_id
    -- , external_id
    -- , fk_client_id
    SELECT *
    FROM (
        -- SELECT *
        -- , ROW_NUMBER() OVER (PARTITION BY fk_commercial_id, client_name ORDER BY NVL(commercial_id, 1) DESC) AS rn
        FROM(
            WITH kin_id AS (
                WITH kintiq_ads AS (
                    SELECT fk_commercial_id
                    FROM prod.detection.commercial_id_external_firehose cief
                    WHERE cief.fk_client_id = 753
                    GROUP BY 1
                )
                , non_kinetiq_ads AS (
                    SELECT cief.fk_commercial_id
                    FROM prod.detection.commercial_id_external_firehose AS cief
                    JOIN kintiq_ads AS b
                        ON cief.fk_commercial_id = b.fk_commercial_id
                    WHERE cief.fk_client_id != 753
                    GROUP BY ALL
                )
                SELECT cief.fk_commercial_id, cief.external_id, cief.fk_client_id
                FROM prod.detection.commercial_id_external_firehose cief
                JOIN non_kinetiq_ads nk
                    ON nk.fk_commercial_id = cief.fk_commercial_id
                GROUP BY ALL
            )
            SELECT kin_id.fk_commercial_id
            , cid.commercial_id
            , cid.commercial_cid
            , kin_id.external_id
            , cl.client_name
            , fim.client_id
            , fim.cid
            , kin_id.fk_client_id
            FROM kin_id
            JOIN prod.detection.clients cl
                ON cl.client_id = kin_id.fk_client_id
            LEFT JOIN prod.public.fileingest_cidmap fim
                ON fim.reportingid = kin_id.external_id
            LEFT JOIN prod.detection.commercial_ids_firehose cid
                ON cid.commercial_cid = SUBSTRING(fim.cid, 20)
            GROUP BY ALL)
            -- WHERE TRUE
            --   AND client_id NOT IN ('kinetiq')
            --   AND client_name = client_id
            --   AND fk_commercial_id != commercial_id
    )
    -- WHERE rn = 1
    ORDER BY 1, 2
)
, combined_set AS (
    SELECT * FROM springserve_ads
    UNION
    SELECT * FROM kinetiq_ads
)
, all_bad_ads AS (
SELECT * FROM combined_set
WHERE fk_client_id NOT IN (753, 1215))
SELECT cid.*
FROM all_bad_ads ba
LEFT JOIN prod.public.cid_map cid
  ON cid.value = ba.external_id
ORDER BY 1, 2, 3, 4

In [0]:
%sql
DESCRIBE prod.detection.viewing_content_golden

In [0]:
column_names_type = {
    "zipcode": "string",
    "dma": "string",
    "tms_episode_id": "string",
    "tivo_episode_id": "string",
    "tms_title": "string",
    "tivo_title": "string",
    "tms_airdate": "timestamp",
    "tivo_airdate": "timestamp",
    "tms_channel_callsign": "string",
    "tivo_channel_callsign": "string",
    "mt_start": "int",
    "tms_channel_affiliate": "string",
    "tivo_channel_affiliate": "string",
    "is_live": "string",
    "ip_address": "string",
    "reported_input_source": "string",
    "content_type": "string",
    "tuner_content_type": "string",
    "input_category": "string",
    "input_device": "string",
    "app_service": "string",
    "tuner_tms_episode_id": "string",
    "tuner_tivo_episode_id": "string",
    "tuner_tms_title": "string",
    "tuner_tivo_title": "string",
    "tuner_tms_airdate": "timestamp",
    "tuner_tivo_airdate": "timestamp",
    "tuner_tms_channel_callsign": "string",
    "tuner_tivo_channel_callsign": "string",
    "tuner_mt_start": "int",
    "tuner_tms_channel_affiliate": "string",
    "tuner_tivo_channel_affiliate": "string",
    "tuner_is_live": "string",
    "tuner_input_category": "string",
    "tuner_input_device": "string",
    "tuner_app_service": "string",
    "tuner_channel_number": "string",
    "enableaudioacr": "string",
    "dma_code": "int",
    "vizio_epg_channel_id": "bigint",
    "vizio_epg_program_id": "bigint",
    "tms_show_genre": "string",
    "tivo_show_genre": "string",
    "tms_epi_title": "string",
    "tivo_epi_title": "string",
    "tuner_tms_epi_title": "string",
    "tuner_tivo_epi_title": "string",
    "series_id": "string",
    "tuner_series_id": "string",
    "show_duration": "int",
    "vizio_epg_not_null": "boolean",
    "nielsen_exclusive": "boolean",
    "content_only_condition": "boolean",
    "tuner_content_only_condition": "boolean",
    "vod_station": "boolean",
    "acrb_clients": "string",
    "appb_clients": "string",
    "client_id_not_null": "string",
    "session_start_hour": "string",
    "tuner_tms_show_genre": "string",
    "tuner_tivo_show_genre": "string",
}

In [0]:
nv_dict = {
    'string': '0',
    'int': 0,
    'bigint': 0,
    'timestamp': '2000-01-01 00:00:00',
    'boolean': 'true'
}

In [0]:
start_time = '2026-08-25 00:00:00'
end_time = '2026-08-26 00:00:00'

In [0]:
for column_name, column_type in column_names_type.items():
    nv = nv_dict.get(column_type, '')

    sql_statement = f"""
        SELECT COUNT(*) as c
        FROM (
        SELECT tvid, session_start, session_end, COUNT(DISTINCT NVL({column_name}, '{nv}')) AS dist_count
        FROM prod.detection.viewing_content_golden
        WHERE session_start_hour >= '{start_time}'
        AND session_start_hour < '{end_time}'
        GROUP BY 1, 2, 3)
        WHERE dist_count > 1;
    """
    x = spark.sql(sql_statement).toPandas().iloc[0]['c']
    print(column_name, x, sep=': ')